In [2]:

from pathlib import Path
import pickle
import networkx as nx

# Find the project root
PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT.name != "adaptive-exam-timetabling"
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

GRAPH_PATH = PROJECT_ROOT / "data" / "processed" / "conflict_graph.gpickle"

with GRAPH_PATH.open("rb") as f:
    G = pickle.load(f)

print("Graph loaded successfully")
print("Vertices:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Project root:", PROJECT_ROOT)


Graph loaded successfully
Vertices: 800
Edges: 9990
Project root: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling


In [3]:

import pandas as pd

def greedy_coloring(graph, order):
    """Colour vertices in the supplied order."""
    coloring = {}

    for node in order:
        used_colors = {
            coloring[neighbor]
            for neighbor in graph.neighbors(node)
            if neighbor in coloring
        }

        color = 0
        while color in used_colors:
            color += 1

        coloring[node] = color

    return coloring


def evaluate_coloring(graph, coloring):
    """Calculate the number of slots and hard-conflict violations."""
    violations = sum(
        1
        for u, v in graph.edges()
        if coloring[u] == coloring[v]
    )

    color_count = len(set(coloring.values()))

    return {
        "Time slots": color_count,
        "Conflict violations": violations,
        "Valid coloring": violations == 0,
    }


# 1. Greedy: alphabetical examination-ID order
greedy_order = sorted(G.nodes())
greedy_result = greedy_coloring(G, greedy_order)

# 2. Welsh–Powell: descending degree order
welsh_powell_order = sorted(
    G.nodes(),
    key=lambda node: (-G.degree(node), node)
)
welsh_powell_result = greedy_coloring(G, welsh_powell_order)

# 3. DSATUR: NetworkX saturation-largest-first strategy
dsatur_result = nx.coloring.greedy_color(
    G,
    strategy="saturation_largest_first"
)

# Compare the results
results = []

for name, coloring in [
    ("Greedy", greedy_result),
    ("Welsh–Powell", welsh_powell_result),
    ("DSATUR", dsatur_result),
]:
    metrics = evaluate_coloring(G, coloring)
    results.append({"Algorithm": name, **metrics})

results_df = pd.DataFrame(results)
display(results_df)


,Algorithm,Time slots,Conflict violations,Valid coloring
0,Greedy,25,0,True
1,Welsh–Powell,18,0,True
2,DSATUR,17,0,True


In [4]:

import pandas as pd
import networkx as nx

# Structural properties of the conflict graph
degrees = dict(G.degree())

graph_summary = {
    "Number of exams": G.number_of_nodes(),
    "Number of conflict edges": G.number_of_edges(),
    "Number of connected components": nx.number_connected_components(G),
    "Maximum degree": max(degrees.values()),
    "Minimum degree": min(degrees.values()),
    "Average degree": round(sum(degrees.values()) / len(degrees), 3),
    "Graph density": round(nx.density(G), 5),
    "Largest component size": max(
        len(component) for component in nx.connected_components(G)
    ),
}

display(pd.DataFrame(
    graph_summary.items(),
    columns=["Metric", "Value"]
))

# Save the current baseline comparison
results_df.to_csv(
    PROJECT_ROOT / "results" / "baseline_comparison.csv",
    index=False
)

print("Baseline results saved.")


,Metric,Value
0,Number of exams,800.00000
1,Number of conflict edges,9990.00000
2,Number of connected components,14.00000
3,Maximum degree,203.00000
4,Minimum degree,0.00000
5,Average degree,24.97500
6,Graph density,0.03126
7,Largest component size,780.00000


Baseline results saved.


In [5]:
components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

print("Connected component sizes:")
for i, component in enumerate(components, start=1):
    print(f"Component {i}: {len(component)} exams")

print("\nExams with no conflicts:")
isolated_exams = list(nx.isolates(G))
print(f"Count: {len(isolated_exams)}")
print(isolated_exams)

Connected component sizes:
Component 1: 780 exams
Component 2: 6 exams
Component 3: 3 exams
Component 4: 1 exams
Component 5: 1 exams
Component 6: 1 exams
Component 7: 1 exams
Component 8: 1 exams
Component 9: 1 exams
Component 10: 1 exams
Component 11: 1 exams
Component 12: 1 exams
Component 13: 1 exams
Component 14: 1 exams

Exams with no conflicts:
Count: 11
['C81J1AE2', 'C81J2AE2', 'C81J3AE2', 'C81J4AE2', 'C81J5AE2', 'C81JIAE2', 'F12X04E2', 'F12X04E3', 'H3BFM2E2', 'Q8P010E1', 'V6B335E1']


In [6]:
# Compare the three baseline graph-colouring algorithms

baseline_results = []

algorithms = {
    "Greedy": greedy_result,
    "Welsh-Powell": welsh_powell_result,
    "DSATUR": dsatur_result,
}

for algorithm_name, coloring in algorithms.items():
    metrics = evaluate_coloring(G, coloring)

    baseline_results.append({
        "Algorithm": algorithm_name,
        **metrics
    })

results_df = pd.DataFrame(baseline_results)

display(results_df)

# Save the baseline comparison
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    results_dir / "baseline_comparison.csv",
    index=False
)

print("Saved baseline results to:", results_dir / "baseline_comparison.csv")

,Algorithm,Time slots,Conflict violations,Valid coloring
0,Greedy,25,0,True
1,Welsh-Powell,18,0,True
2,DSATUR,17,0,True


Saved baseline results to: c:\Users\Nethmi\Documents\GitHub\adaptive-exam-timetabling\results\baseline_comparison.csv


In [7]:
# Inspect the number of colours assigned by each baseline

for name, coloring in {
    "Greedy": greedy_result,
    "Welsh-Powell": welsh_powell_result,
    "DSATUR": dsatur_result,
}.items():
    color_counts = pd.Series(coloring).value_counts().sort_index()

    print(f"\n{name}")
    print(f"Number of colours: {len(color_counts)}")
    print(f"Smallest colour class: {color_counts.min()} exams")
    print(f"Largest colour class: {color_counts.max()} exams")
    print(f"Average exams per colour: {color_counts.mean():.2f}")


Greedy
Number of colours: 25
Smallest colour class: 1 exams
Largest colour class: 166 exams
Average exams per colour: 32.00

Welsh-Powell
Number of colours: 18
Smallest colour class: 3 exams
Largest colour class: 135 exams
Average exams per colour: 44.44

DSATUR
Number of colours: 17
Smallest colour class: 2 exams
Largest colour class: 145 exams
Average exams per colour: 47.06


In [8]:
from pathlib import Path

data_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "data"

with open(data_file, "r", encoding="utf-8", errors="replace") as f:
    data_lines = f.readlines()

print("Total lines in original data file:", len(data_lines))

print("\nLines mentioning consecutive exams, gaps, or spacing:")
keywords = ("consecutive", "gap", "immediately", "one period", "overnight")

for line_number, line in enumerate(data_lines, start=1):
    if any(word in line.lower() for word in keywords):
        print(f"Line {line_number}: {line.rstrip()}")

Total lines in original data file: 182

Lines mentioning consecutive exams, gaps, or spacing:
Line 146: All students must have a one-period (or overnight) gap between consecutive
Line 165: H3BFM2E1  must be immediately followed by  H3BFM2E2


In [9]:
# Display the original timetable constraints with line numbers

for line_number in range(140, 183):
    if line_number <= len(data_lines):
        print(f"{line_number}: {data_lines[line_number - 1].rstrip()}")

140: H3BDM3E1  50
141: HGAEM2E1  50
142: HGBEM3E1  50
143: 
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan
152: H22M02E1  23rd-24th Jan
153: H2CM04E1  23rd-24th Jan
154: 
155: G13RE2E1  before 30th Jan
156: 
157: K1AHWAE2  any am slot
158: H63122E1  any am slot
159: 
160: V13101E1  any Thursday pm slot
161: 
162: F13P03E1 \ before / F13X03E1
163: F13P05E1 /        \ F13X04E1
164: 
165: H3BFM2E1  must be immediately followed by  H3BFM2E2
166: 
167: H8B040E1 \ must be at different times
168: H8C001E1 /
169: 
170: Q33211E1 \
171: Q33307E1 - spread out, mid-session
172: Q33308E1 /
173: 
174: B12301E1 \
175: B12302E1 - spread out
176: B12303E1 |
177: B12320E1 /
178: 
179: B13103E1              \ spread out
180: {C13571E1 & C13572E1} /
181: 
182: Room TRENT-B46 unavailable morning of Friday 3rd Feb


In [10]:
# Inspect the beginning of the benchmark data file
# to locate examination dates, times, and room definitions.

for line_number, line in enumerate(data_lines[:80], start=1):
    print(f"{line_number}: {line.rstrip()}")

1: DATES
2: -----
3: Mon 23rd Jan - Sat 4th Feb 1995
4: 
5: TIMES
6: -----
7: Mon - Fri  9:00 (3hrs), 13:30 (2hrs), 16:30 (2hrs)
8: Sat        9:00 (3hrs)
9: 
10: ROOMS
11: -----
12: TRENT-HALL   125
13: TRENT-L19     80
14: TRENT-B46     40
15: COPSE-2       40
16: COPSE-3       50
17: COPSE-4       35
18: COPSE-5       30
19: COPSE-6       25
20: PRTLND-GRND  200
21: POPE-A13      80 \ together  \
22: POPE-A14      80 /           |  Rooms that are together can be combined,
23: ART-LECTURE   80 \ together  -  ie. used as one room.  This is not as good as
24: ART-SEMINAR   15 /           |  using one room, but is better than splitting
25: SPORT-LGE1   250 \ together  |  an exam over two distant rooms.
26: SPORT-LGE2   230 /           /
27: SPORT-SMALL  270
28: 
29: ROOM ASSIGNMENTS
30: ----------------
31: V41103E1   ART-SEMINAR
32: V41112E1   ART-LECTURE
33: V41124E1   ART-SEMINAR
34: V41133E1   ART-LECTURE
35: V41142E1   ART-LECTURE
36: V41143E1   ART-SEMINAR
37: V42103E1   ART-LECTU

In [11]:
from datetime import date, timedelta

start_date = date(1995, 1, 23)
end_date = date(1995, 2, 4)

periods = []

current_date = start_date

while current_date <= end_date:
    if current_date.weekday() < 5:  # Monday to Friday
        times = ["09:00", "13:30", "16:30"]
    else:
        times = ["09:00"] if current_date.weekday() == 5 else []

    for time_slot in times:
        periods.append({
            "period_index": len(periods),
            "date": current_date.isoformat(),
            "day": current_date.strftime("%A"),
            "time": time_slot,
        })

    current_date += timedelta(days=1)

periods_df = pd.DataFrame(periods)

display(periods_df)
print("\nTotal available periods:", len(periods_df))

,period_index,date,day,time
0,0,1995-01-23,Monday,09:00
1,1,1995-01-23,Monday,13:30
2,2,1995-01-23,Monday,16:30
3,3,1995-01-24,Tuesday,09:00
4,4,1995-01-24,Tuesday,13:30
5,5,1995-01-24,Tuesday,16:30
6,6,1995-01-25,Wednesday,09:00
7,7,1995-01-25,Wednesday,13:30
8,8,1995-01-25,Wednesday,16:30
9,9,1995-01-26,Thursday,09:00



Total available periods: 32


In [12]:
# Validate the generated examination period calendar

assert len(periods_df) == 32, "Unexpected number of periods"
assert periods_df["period_index"].tolist() == list(range(32))
assert not periods_df.duplicated(["date", "time"]).any()

print("Period count:", len(periods_df))
print("Unique dates:", periods_df["date"].nunique())
print("Unique date-time combinations:",
      periods_df[["date", "time"]].drop_duplicates().shape[0])

print("\nPeriods per date:")
display(
    periods_df.groupby(["date", "day"]).size()
    .reset_index(name="number_of_periods")
)

print("\nCalendar validation passed.")

Period count: 32
Unique dates: 12
Unique date-time combinations: 32

Periods per date:


,date,day,number_of_periods
0,1995-01-23,Monday,3
1,1995-01-24,Tuesday,3
2,1995-01-25,Wednesday,3
3,1995-01-26,Thursday,3
4,1995-01-27,Friday,3
5,1995-01-28,Saturday,1
6,1995-01-30,Monday,3
7,1995-01-31,Tuesday,3
8,1995-02-01,Wednesday,3
9,1995-02-02,Thursday,3



Calendar validation passed.


In [13]:
# Map DSATUR colour labels to the first available examination periods

dsatur_colors = sorted(set(dsatur_result.values()))

color_to_period = {
    color: period_index
    for period_index, color in enumerate(dsatur_colors)
}

dsatur_schedule = pd.DataFrame({
    "exam_id": list(dsatur_result.keys()),
    "color": list(dsatur_result.values()),
})

dsatur_schedule["period_index"] = (
    dsatur_schedule["color"].map(color_to_period)
)

dsatur_schedule = dsatur_schedule.merge(
    periods_df,
    on="period_index",
    how="left",
)

print("Exams scheduled:", len(dsatur_schedule))
print("Unique periods used:", dsatur_schedule["period_index"].nunique())
print(
    "Exams per assigned period:",
    dsatur_schedule.groupby("period_index").size().to_dict()
)

display(dsatur_schedule.head(10))

Exams scheduled: 800
Unique periods used: 17
Exams per assigned period: {0: 145, 1: 108, 2: 92, 3: 77, 4: 67, 5: 58, 6: 52, 7: 49, 8: 46, 9: 33, 10: 20, 11: 17, 12: 14, 13: 11, 14: 5, 15: 4, 16: 2}


,exam_id,color,period_index,date,day,time
0,LKA1FAE1,0,0,1995-01-23,Monday,09:00
1,LK10SAE1,1,1,1995-01-23,Monday,13:30
2,L81122E1,2,2,1995-01-23,Monday,16:30
3,N1A110E1,3,3,1995-01-24,Tuesday,09:00
4,C81ICAE1,4,4,1995-01-24,Tuesday,13:30
5,L81112E1,5,5,1995-01-24,Tuesday,16:30
6,V11111E1,6,6,1995-01-25,Wednesday,09:00
7,M11347E1,7,7,1995-01-25,Wednesday,13:30
8,L11101E1,8,8,1995-01-25,Wednesday,16:30
9,L11100E1,9,9,1995-01-26,Thursday,09:00


In [14]:

from collections import defaultdict
from pathlib import Path

# Locate the project root and enrolments file
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data" / "raw" / "Nott").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate data/raw/Nott")
    PROJECT_ROOT = PROJECT_ROOT.parent

enrolments_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "enrolements"

# Reconstruct student-exam pairs from the fixed-width file
student_exam_pairs = []

with open(enrolments_file, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\r\n")

        if len(line) >= 18:
            student_id = line[:10].strip()
            exam_id = line[10:18].strip()

            if student_id and exam_id:
                student_exam_pairs.append((student_id, exam_id))

print("Student-exam pairs loaded:", len(student_exam_pairs))

# Build the student-to-exam mapping
student_to_exams = defaultdict(list)

for student_id, exam_id in student_exam_pairs:
    student_to_exams[student_id].append(exam_id)

# Map each exam to its assigned period
exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

# Find pairs of exams assigned to adjacent periods
consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted(
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
        if exam_id in exam_to_period
    )

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print(
    "Student exam pairs in consecutive periods:",
    len(consecutive_exam_pairs)
)

print("Sample pairs:")
for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Student-exam pairs loaded: 33997
Student exam pairs in consecutive periods: 0
Sample pairs:


In [15]:

# Inspect the number of distinct periods used by each student

student_periods = {}

for student_id, exams_for_student in student_to_exams.items():
    student_periods[student_id] = sorted({
        exam_to_period[exam_id]
        for exam_id in exams_for_student
        if exam_id in exam_to_period
    })

period_gaps = []

for student_id, assigned_periods in student_periods.items():
    for i in range(len(assigned_periods) - 1):
        gap = assigned_periods[i + 1] - assigned_periods[i]
        period_gaps.append(gap)

print("Students checked:", len(student_periods))
print("Student period gaps checked:", len(period_gaps))

if period_gaps:
    print("Minimum period gap:", min(period_gaps))
    print("Maximum period gap:", max(period_gaps))
    print("Number of adjacent-period gaps:", sum(gap == 1 for gap in period_gaps))
    print("Number of gaps of two periods:", sum(gap == 2 for gap in period_gaps))
else:
    print("No period gaps were available to inspect.")


Students checked: 7896
Student period gaps checked: 0
No period gaps were available to inspect.


In [16]:
# Diagnose whether enrolment exam IDs match the DSATUR schedule

enrolment_exam_ids = {exam_id for _, exam_id in student_exam_pairs}
scheduled_exam_ids = set(dsatur_schedule["exam_id"])

print("Unique exam IDs in enrolments:", len(enrolment_exam_ids))
print("Unique exam IDs in schedule:", len(scheduled_exam_ids))
print("Matching exam IDs:", len(enrolment_exam_ids & scheduled_exam_ids))
print("Enrolment exam IDs not scheduled:", len(enrolment_exam_ids - scheduled_exam_ids))
print("Scheduled exam IDs not in enrolments:", len(scheduled_exam_ids - enrolment_exam_ids))

# Show a few examples if IDs do not match
print("\nUnmatched enrolment exam IDs:")
print(sorted(enrolment_exam_ids - scheduled_exam_ids)[:10])

print("\nSample scheduled exam IDs:")
print(sorted(scheduled_exam_ids)[:10])


Unique exam IDs in enrolments: 791
Unique exam IDs in schedule: 800
Matching exam IDs: 0
Enrolment exam IDs not scheduled: 791
Scheduled exam IDs not in enrolments: 800

Unmatched enrolment exam IDs:
['AA2016E', 'AA3008E', 'AAA011E', 'AAA013E', 'AAA022E', 'AAA024E', 'AAA033E', 'AAA034E', 'AAB025E', 'AAB058E']

Sample scheduled exam IDs:
['AA2016E1', 'AA3008E1', 'AAA011E1', 'AAA013E1', 'AAA022E1', 'AAA024E1', 'AAA033E1', 'AAA034E1', 'AAB025E1', 'AAB058E1']


In [17]:

# Normalize enrolment exam IDs against the actual schedule IDs

scheduled_exam_ids = set(dsatur_schedule["exam_id"])

student_to_exams = defaultdict(list)

unmatched_exam_ids = set()

for student_id, exam_id in student_exam_pairs:
    if exam_id in scheduled_exam_ids:
        normalized_exam_id = exam_id
    elif exam_id + "1" in scheduled_exam_ids:
        normalized_exam_id = exam_id + "1"
    else:
        unmatched_exam_ids.add(exam_id)
        continue

    student_to_exams[student_id].append(normalized_exam_id)

print("Students mapped:", len(student_to_exams))
print("Unmatched unique enrolment exam IDs:", len(unmatched_exam_ids))
print("Unmatched examples:", sorted(unmatched_exam_ids)[:10])

# Recalculate adjacent-period exam pairs
exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted({
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
    })

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print(
    "Student exam pairs in consecutive periods:",
    len(consecutive_exam_pairs)
)
print("Sample pairs:")
for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Students mapped: 7896
Unmatched unique enrolment exam IDs: 1
Unmatched examples: ['K1AHWAE']
Student exam pairs in consecutive periods: 12568
Sample pairs:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)
('A903428732', 'R13003E1', 'R13021E1', 0, 1)
('A903428732', 'R13021E1', 'R13001E1', 1, 2)
('A903549679', 'V73128E1', 'V73127E1', 3, 4)
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A904679991', 'Q83184E1', 'Q8B094E1', 0, 1)


In [18]:

# Inspect the unmatched exam in the original exam and enrolment files

exams_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "exams"

print("Matching records in the exams file:")
with open(exams_file, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        if "K1AHWA" in line:
            print(repr(line))

print("\nMatching enrolment exam IDs:")
for exam_id in sorted({
    exam_id for _, exam_id in student_exam_pairs
}):
    if "K1AHWA" in exam_id:
        print(repr(exam_id))

print("\nMatching scheduled exam IDs:")
for exam_id in sorted(scheduled_exam_ids):
    if "K1AHWA" in exam_id:
        print(repr(exam_id))


Matching records in the exams file:
'K1AHWAE2 HISTORY OF WESTERN ARCHITECTURE          0:50 AR\n'

Matching enrolment exam IDs:
'K1AHWAE'

Matching scheduled exam IDs:
'K1AHWAE2'


In [19]:

# Investigate the unmatched K1AHWAE identifier

unmatched_student_ids = {
    student_id
    for student_id, exam_id in student_exam_pairs
    if exam_id == "K1AHWAE"
}

print("Students enrolled under K1AHWAE:", len(unmatched_student_ids))
print("K1AHWAE2 exists in schedule:", "K1AHWAE2" in scheduled_exam_ids)

print("\nSample student IDs:")
for student_id in sorted(unmatched_student_ids)[:10]:
    print(student_id)


Students enrolled under K1AHWAE: 58
K1AHWAE2 exists in schedule: True

Sample student IDs:
A940013331
A940015542
A940023916
A940025596
B940012884
B940013548
B940014785
B940014844
B940014903
B940014925


In [20]:

# Normalize enrolment exam IDs using verified source-data rules

scheduled_exam_ids = set(dsatur_schedule["exam_id"])

student_to_exams = defaultdict(list)
unmatched_exam_ids = set()

for student_id, original_exam_id in student_exam_pairs:
    if original_exam_id in scheduled_exam_ids:
        normalized_exam_id = original_exam_id

    elif original_exam_id == "K1AHWAE" and "K1AHWAE2" in scheduled_exam_ids:
        # Explicit exception verified against the original exams file
        normalized_exam_id = "K1AHWAE2"

    elif original_exam_id + "1" in scheduled_exam_ids:
        # Standard suffix normalization, verified against scheduled IDs
        normalized_exam_id = original_exam_id + "1"

    else:
        unmatched_exam_ids.add(original_exam_id)
        continue

    student_to_exams[student_id].append(normalized_exam_id)

print("Students mapped:", len(student_to_exams))
print("Unmatched unique exam IDs:", len(unmatched_exam_ids))
print("Unmatched IDs:", sorted(unmatched_exam_ids))
print("Total enrolment records mapped:", sum(map(len, student_to_exams.values())))

assert len(student_to_exams) == 7896
assert len(unmatched_exam_ids) == 0
assert sum(map(len, student_to_exams.values())) == 33997

print("Identifier normalization checks passed.")


Students mapped: 7896
Unmatched unique exam IDs: 0
Unmatched IDs: []
Total enrolment records mapped: 33997
Identifier normalization checks passed.


In [21]:

# Recalculate consecutive-period pairs with corrected exam IDs

exam_to_period = dict(
    zip(dsatur_schedule["exam_id"], dsatur_schedule["period_index"])
)

consecutive_exam_pairs = set()

for student_id, exams_for_student in student_to_exams.items():
    assigned = sorted({
        (exam_to_period[exam_id], exam_id)
        for exam_id in exams_for_student
    })

    for i in range(len(assigned) - 1):
        period_a, exam_a = assigned[i]
        period_b, exam_b = assigned[i + 1]

        if period_b == period_a + 1:
            consecutive_exam_pairs.add(
                (student_id, exam_a, exam_b, period_a, period_b)
            )

print("Students evaluated:", len(student_to_exams))
print("Adjacent-period student exam pairs:", len(consecutive_exam_pairs))
print("Sample pairs:")

for pair in sorted(consecutive_exam_pairs)[:10]:
    print(pair)


Students evaluated: 7896
Adjacent-period student exam pairs: 12574
Sample pairs:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)
('A903428732', 'R13003E1', 'R13021E1', 0, 1)
('A903428732', 'R13021E1', 'R13001E1', 1, 2)
('A903549679', 'V73128E1', 'V73127E1', 3, 4)
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A904679991', 'Q83184E1', 'Q8B094E1', 0, 1)


In [22]:

# Display the original benchmark spacing and scheduling constraints

data_file = PROJECT_ROOT / "data" / "raw" / "Nott" / "data"

with open(data_file, "r", encoding="utf-8", errors="replace") as f:
    data_lines = f.readlines()

for i, line in enumerate(data_lines):
    if "MISC" in line or "All students must have" in line:
        start = max(0, i - 2)
        end = min(len(data_lines), i + 8)

        print(f"Constraint lines {start + 1}–{end}:")
        for j in range(start, end):
            print(f"{j + 1}: {data_lines[j].rstrip()}")

        print()


Constraint lines 142–151:
142: HGBEM3E1  50
143: 
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan

Constraint lines 144–153:
144: MISC
145: ----
146: All students must have a one-period (or overnight) gap between consecutive
147: exams.
148: 
149: F321Q6E1  27th Jan
150: F321T6E1  30th Jan
151: H21M01E1  23rd-24th Jan
152: H22M02E1  23rd-24th Jan
153: H2CM04E1  23rd-24th Jan



In [23]:

# Inspect chronological examination periods and their transitions

calendar_check = periods_df.sort_values("period_index").copy()

calendar_check["next_date"] = calendar_check["date"].shift(-1)
calendar_check["next_day"] = calendar_check["day"].shift(-1)
calendar_check["next_time"] = calendar_check["time"].shift(-1)

calendar_check["transition"] = calendar_check.apply(
    lambda row: (
        "End of calendar"
        if row["next_date"] is None or pd.isna(row["next_date"])
        else "Same day"
        if row["date"] == row["next_date"]
        else "Overnight / next date"
    ),
    axis=1,
)

display(
    calendar_check[
        ["period_index", "date", "day", "time",
         "next_date", "next_day", "next_time", "transition"]
    ]
)

print("\nTransition counts:")
print(calendar_check["transition"].value_counts())


,period_index,date,day,time,next_date,next_day,next_time,transition
0,0,1995-01-23,Monday,09:00,1995-01-23,Monday,13:30,Same day
1,1,1995-01-23,Monday,13:30,1995-01-23,Monday,16:30,Same day
2,2,1995-01-23,Monday,16:30,1995-01-24,Tuesday,09:00,Overnight / next date
3,3,1995-01-24,Tuesday,09:00,1995-01-24,Tuesday,13:30,Same day
4,4,1995-01-24,Tuesday,13:30,1995-01-24,Tuesday,16:30,Same day
5,5,1995-01-24,Tuesday,16:30,1995-01-25,Wednesday,09:00,Overnight / next date
6,6,1995-01-25,Wednesday,09:00,1995-01-25,Wednesday,13:30,Same day
7,7,1995-01-25,Wednesday,13:30,1995-01-25,Wednesday,16:30,Same day
8,8,1995-01-25,Wednesday,16:30,1995-01-26,Thursday,09:00,Overnight / next date
9,9,1995-01-26,Thursday,09:00,1995-01-26,Thursday,13:30,Same day



Transition counts:
transition
Same day                 20
Overnight / next date    11
End of calendar           1
Name: count, dtype: int64


In [24]:

# Classify adjacent-period student exam pairs by calendar transition

period_lookup = periods_df.set_index("period_index").to_dict("index")

same_day_pairs = []
different_date_pairs = []

for student_id, exam_a, exam_b, period_a, period_b in consecutive_exam_pairs:
    date_a = period_lookup[period_a]["date"]
    date_b = period_lookup[period_b]["date"]

    pair = (student_id, exam_a, exam_b, period_a, period_b)

    if date_a == date_b:
        same_day_pairs.append(pair)
    else:
        different_date_pairs.append(pair)

print("Adjacent-period student exam pairs:", len(consecutive_exam_pairs))
print("Same-day adjacent pairs:", len(same_day_pairs))
print("Different-date adjacent pairs:", len(different_date_pairs))

print("\nSame-day examples:")
for pair in sorted(same_day_pairs)[:5]:
    print(pair)

print("\nDifferent-date examples:")
for pair in sorted(different_date_pairs)[:5]:
    print(pair)


Adjacent-period student exam pairs: 12574
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944

Same-day examples:
('A890186790', 'R13006E1', 'R13021E1', 0, 1)
('A890186790', 'R13021E1', 'R13001E1', 1, 2)
('A890186790', 'R13022E1', 'R13016E1', 7, 8)
('A891097581', 'R23107E1', 'R23117E1', 7, 8)
('A891097581', 'R23111E1', 'R23107E1', 6, 7)

Different-date examples:
('A903549679', 'V73232E1', 'V73128E1', 2, 3)
('A905947446', 'R23102E1', 'R13009E1', 2, 3)
('A913122273', 'V83123E1', 'R23111E1', 5, 6)
('A913360369', 'R23117E1', 'R13011E1', 8, 9)
('A913765265', 'R83044E1', 'R83038E1', 2, 3)


In [29]:

# Reusable diagnostic for student exam spacing

def evaluate_spacing(exam_to_period, student_to_exams, periods_df):
    period_lookup = periods_df.set_index("period_index").to_dict("index")

    same_day_adjacent = []
    different_date_adjacent = []
    same_period_conflicts = []

    for student_id, exams in student_to_exams.items():
        # Remove duplicate exam entries for each student
        unique_exams = set(exams)

        assigned = sorted(
            (exam_to_period[exam_id], exam_id)
            for exam_id in unique_exams
            if exam_id in exam_to_period
        )

        # Check all distinct exam pairs for same-period conflicts
        for i in range(len(assigned)):
            for j in range(i + 1, len(assigned)):
                period_a, exam_a = assigned[i]
                period_b, exam_b = assigned[j]

                if period_a == period_b:
                    same_period_conflicts.append(
                        (student_id, exam_a, exam_b, period_a)
                    )

        # Check consecutive exams for adjacent-period spacing
        for i in range(len(assigned) - 1):
            period_a, exam_a = assigned[i]
            period_b, exam_b = assigned[i + 1]

            if period_b == period_a + 1:
                date_a = period_lookup[period_a]["date"]
                date_b = period_lookup[period_b]["date"]

                pair = (
                    student_id, exam_a, exam_b,
                    period_a, period_b
                )

                if date_a == date_b:
                    same_day_adjacent.append(pair)
                else:
                    different_date_adjacent.append(pair)

    return {
        "same_period_conflicts": same_period_conflicts,
        "same_day_adjacent": same_day_adjacent,
        "different_date_adjacent": different_date_adjacent,
        "same_period_conflict_count": len(same_period_conflicts),
        "same_day_adjacent_count": len(same_day_adjacent),
        "different_date_adjacent_count": len(different_date_adjacent),
    }


# Evaluate the current DSATUR assignment
spacing_results = evaluate_spacing(
    exam_to_period=exam_to_period,
    student_to_exams=student_to_exams,
    periods_df=periods_df,
)

print("Same-period conflicts:",
      spacing_results["same_period_conflict_count"])
print("Same-day adjacent pairs:",
      spacing_results["same_day_adjacent_count"])
print("Different-date adjacent pairs:",
      spacing_results["different_date_adjacent_count"])


Same-period conflicts: 0
Same-day adjacent pairs: 8630
Different-date adjacent pairs: 3944


In [31]:

# Investigate same-period conflicts

print("Schedule rows:", len(dsatur_schedule))
print("Unique scheduled exams:", dsatur_schedule["exam_id"].nunique())
print("Duplicate exam IDs:", dsatur_schedule["exam_id"].duplicated().sum())
print("Unique graph vertices:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

conflicts = spacing_results["same_period_conflicts"]

print("\nNumber of reported same-period conflicts:", len(conflicts))
print("First 15 conflicting pairs:")

for conflict in conflicts[:15]:
    student_id, exam_a, exam_b, period = conflict

    print(
        f"Student={student_id}, "
        f"Exam A={exam_a}, Exam B={exam_b}, "
        f"Period={period}, "
        f"Color A={dsatur_result.get(exam_a)}, "
        f"Color B={dsatur_result.get(exam_b)}"
    )


Schedule rows: 800
Unique scheduled exams: 800
Duplicate exam IDs: 0
Unique graph vertices: 800
Graph edges: 9990

Number of reported same-period conflicts: 0
First 15 conflicting pairs:
